# MIPS Hardness and the Limits of Exact Nearest-Neighbor Search — companion notebook

This notebook is the **narrative** half of the topic's Python pillar. The reusable,
tested implementation lives next door in
[`mips_hardness_and_sublinearity_limits.py`](mips_hardness_and_sublinearity_limits.py);
here we import it and walk the topic section by section. The `.py` *owns the numbers* —
its `assert`-based harness encodes each claim, and running it as a script is the
regression test.

```bash
uv run --with numpy python notebooks/mips-hardness-and-sublinearity-limits/mips_hardness_and_sublinearity_limits.py
```


In [ ]:
import sys
import pathlib

_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "mips-hardness-and-sublinearity-limits",
              pathlib.Path("notebooks/mips-hardness-and-sublinearity-limits")):
    if (_cand / "mips_hardness_and_sublinearity_limits.py").exists():
        sys.path.insert(0, str(_cand))
        break

import numpy as np
import mips_hardness_and_sublinearity_limits as mips


## 1. MIPS is not a metric problem

Maximum inner-product search returns $\operatorname{arg\,max}_i \langle q, C_i\rangle$.
It is *not* the metric nearest-neighbor problem it is often mistaken for: the inner
product has no triangle inequality, and a vector need not even be its own best match
— a longer vector in the same direction scores higher. So the metric-ball intuition
that organizes tree and graph indexes does not transfer.


In [ ]:
mips.test_brute_force_crosscheck()
mips.test_mips_not_self_similar()


## 2. The asymmetric lifting transform: MIPS → Euclidean nearest neighbor

Append one coordinate so every database vector has equal norm $M$, and a zero
coordinate to the query. Then $\lVert \tilde q - \tilde p\rVert^2 = \lVert q\rVert^2 + M^2 - 2\langle q, p\rangle$,
so the lifted Euclidean nearest neighbor is the exact MIPS winner. The harness checks
this on 200 random queries.


In [ ]:
mips.test_lifting_preserves_argmax()

## 3. The lifting preserves the argmax but distorts approximation ratios

The exact winner is preserved, but a $c$-approximate Euclidean neighbor in the lifted
space is **not** a $c$-approximate inner product: the additive constant
$\lVert q\rVert^2 + M^2$ does not pass through the multiplicative ratio. The runner-up
$B$ below is a good MIPS approximation ($1.06\times$) yet a poor lifted-distance
approximation ($1.73\times$) — the two factors straddle $c = 1.3$.


In [ ]:
r = mips.lifting_distortion()
print(f"M = {r['M']:.5f}")
print(f"<q,A> = {r['s_A']:.4f}   <q,B> = {r['s_B']:.4f}")
print(f"lifted dist to A = {r['dist_A']:.5f}   to B = {r['dist_B']:.5f}")
print(f"MIPS factor s_A/s_B = {r['mips_factor']:.4f}   NN factor distB/distA = {r['nn_factor']:.4f}")
mips.test_lifting_distorts_approximation_ratio()


## 4. The hardness core: Orthogonal Vectors → farthest pair

With fixed Hamming weight $w$, two Boolean vectors satisfy
$\lVert a-b\rVert^2 = 2w - 2\langle a,b\rangle$, so an **orthogonal** pair attains the
maximal squared distance $2w$ and every non-orthogonal cross pair is at most $2w-2$ —
a clean gap of $2$. A truly-subquadratic exact farthest-pair algorithm would therefore
decide Orthogonal Vectors, which under the Strong Exponential Time Hypothesis (via
Williams 2005) requires $n^{2-o(1)}$ time. This is the load-bearing reduction.


In [ ]:
mips.test_ov_distance_gap()

## 5. The empirical curse: metric pruning collapses

A triangle-inequality pivot filter prunes a candidate $x$ when some pivot $p$ certifies
$|D(q,p) - D(p,x)| > r$ for the current best radius $r$. As the ambient dimension grows,
distances concentrate (the high-dimensional-geometry topic proves this), the certificates
vanish, and the **inspected fraction climbs toward 1** — exact search degenerates into a
linear scan. The grid below drives the laboratory's dimension slider.


In [ ]:
for d, f in mips.grid_table():
    print(f"d = {d:>4}:  inspected fraction {f:.4f}")
mips.test_pruning_collapses()


## 6. Finance case study: exact MIPS is $O(nd)$ per query

On the production multimodal corpus (filings, transcripts, charts embedded in
$\mathbb{R}^{1536}$), an exact MIPS scan costs $2nd$ multiply-adds per query and grows
linearly in the corpus size — untenable at millions of vectors, which is exactly why the
rest of the curriculum builds approximate indexes.


In [ ]:
mips.test_finance_scaling_is_linear()
mips.finance_demo()


---

Every number above is asserted in
[`mips_hardness_and_sublinearity_limits.py`](mips_hardness_and_sublinearity_limits.py):
the non-self-similarity of MIPS, the argmax-preserving lift and its ratio distortion, the
Orthogonal-Vectors distance gap, the pruning collapse, and the linear-in-$n$ scan cost.
The three pillars — the proofs on the page, the interactive laboratory, and this tested
code — agree by construction.
